# Persuasion / Goal-Influencing Dialogue — Demo (issue #46)

Characters can **talk**, and what they hear can **change what they want** — but only if it
fits who they are. A speaker's utterance reaches co-located listeners (a bounded `heard`
buffer); a listener's agent may then **adopt** or **drop** a goal through the normal
precondition gate. What keeps persuasion honest is the deciding agent's **persona**, not a
restriction on the action — a stubborn knight can refuse the very same words a loyal
servant obeys.

This is distinct from **knowledge** (#45, see `04_knowledge_beliefs.ipynb`): knowledge is
what a character *believes is true*; this is about *speech moving goals*.

Fully **offline and deterministic** — driven by the `MockReActClient`, no API key. It
shows:

1. Speaking — broadcast vs. directed, and **who hears it**.
2. Audience is **room-scoped**.
3. The **heard buffer** in an agent's observation.
4. **Adopt / drop goal** through the precondition gate.
5. **Persona is the gate** — the same request, two different outcomes.

## 1. Speaking, and who hears it

`say <message>` broadcasts to the room; `say to <name> <message>` directs it at a
co-located character. Either way, listeners record the line in their `heard` buffer,
phrased from *their* point of view ("said to you" vs. "said:"). The speaker never hears
their own voice.

In [1]:
from text_adventure_games import games, things


def two_char_room():
    field = things.Location("Field", "A grassy field.")
    alice = things.Character("alice", "a herald", "I speak.")
    bob = things.Character("bob", "a fellow", "I listen.")
    game = games.Game(field, alice, characters=[bob])
    field.add_character(bob)
    return game, alice, bob


game, alice, bob = two_char_room()
game.parser.parse_command("say to bob fetch the key", actor=alice)
print("after directed speech, bob heard:", bob.heard)

game.parser.parse_command("say hello everyone", actor=alice)
print("after broadcast,        bob heard:", bob.heard)
print("does the speaker hear herself? ->", alice.heard)

alice says to bob: fetch the key
after directed speech, bob heard: ['alice said to you: fetch the key']
alice says: hello everyone
after broadcast,        bob heard: ['alice said to you: fetch the key', 'alice said: hello everyone']
does the speaker hear herself? -> []


## 2. Audience is room-scoped

Who hears is decided by `Game.audience_for(speaker, message, target)` — room-based by
default (override it for a continuous / range-based world). A listener in another room
hears nothing.

In [2]:
field = things.Location("Field", "A field.")
forest = things.Location("Forest", "A forest.")
field.add_connection("north", forest)
alice = things.Character("alice", "a herald", "I speak.")
carol = things.Character("carol", "afar", "I am elsewhere.")
game = games.Game(field, alice, characters=[carol])
forest.add_character(carol)  # NOT co-located with alice

game.parser.parse_command("say hello", actor=alice)
print("audience for alice's shout:", game.audience_for(alice, "hello"))
print("carol (in the forest) heard:", carol.heard)

alice says: hello
audience for alice's shout: []
carol (in the forest) heard: []


## 3. The heard buffer in the observation

When an agent decides, its observation includes a **"You recently heard:"** section plus a
note that heard speech is *optional input* — it should sway behavior only when it genuinely
fits the persona. That note is what keeps persuasion non-automatic. The buffer is FIFO and
capped at `HEARD_MAX`.

In [3]:
from text_adventure_games.npc import build_npc_context

game, alice, bob = two_char_room()
bob.hear("alice said to you: fetch the key")
print(build_npc_context(bob, game))

FIELD
A grassy field.
Characters here:
 * alice - a herald
Inventory: empty
Available actions: adopt goal, attack, catch fish, describe, drink, drop, drop goal, eat, examine, get, give, go, inventory, light, pick rose, quit, say, sequence, smell rose, take off, unwield, wait, wear, wield
Turn: 0

You recently heard:
  - alice said to you: fetch the key
Not everything you hear matters. Speech may be irrelevant, idle, or contrary to who you are -- only adopt or drop a goal if it genuinely fits your persona and what you already want. Otherwise, ignore it and act normally.


In [4]:
from text_adventure_games.things.characters import HEARD_MAX

c = things.Character("c", "a listener", "I listen.")
for i in range(HEARD_MAX + 3):
    c.hear(f"line {i}")
print(f"HEARD_MAX = {HEARD_MAX}; only the newest are kept:")
print(c.heard)

HEARD_MAX = 5; only the newest are kept:
['line 3', 'line 4', 'line 5', 'line 6', 'line 7']


## 4. Adopt / drop a goal — through the precondition gate

`adopt goal <text>` and `drop goal <text>` are ordinary actions: they pass through the
same precondition gate as everything else (empty text fails, adopting a goal you already
hold fails, dropping one you don't hold fails). An adopted goal is **SHORT-term** — an
intention prompted by the moment.

In [5]:
from text_adventure_games.actions.goals import AdoptGoal

field = things.Location("Field", "A field.")
bob = things.Character("bob", "a fellow", "I listen.")
game = games.Game(field, bob)

game.parser.parse_command("adopt goal fetch the key", actor=bob)
print("goals after adopt:", [g.description for g in bob.goals])

# The precondition gate refuses a duplicate.
dup_ok = AdoptGoal(game, "adopt goal fetch the key", actor=bob).check_preconditions()
print("adopting the same goal again allowed?", dup_ok)

game.parser.parse_command("drop goal fetch the key", actor=bob)
print("goals after drop: ", [g.description for g in bob.goals])

bob adopts a new goal: fetch the key
goals after adopt: ['fetch the key']
bob already has that goal.
adopting the same goal again allowed? False
bob drops the goal: fetch the key
goals after drop:  []


## 5. Persona is the gate — persuasion end-to-end

Now the whole loop. A **master** (the player) asks for a favor; on the NPC's turn it hears
the request and decides. Same scene, same words — but a **loyal servant** adopts the goal
while a **stubborn knight** refuses. The only difference is the persona each agent is given.

In [6]:
from text_adventure_games.llm_client import MockReActClient
from text_adventure_games.npc import make_react_behavior


def persuasion_scene(name, persona):
    field = things.Location("Field", "A grassy field.")
    master = things.Character("master", "a noble", "I command my servants.")
    npc = things.Character(name, "a retainer", persona)
    game = games.Game(field, master, characters=[npc])
    field.add_character(npc)
    npc.set_behavior(make_react_behavior(MockReActClient()))
    return game, master, npc


# The player speaks; do_command then gives the NPC its turn, on which it hears
# the request and decides whether to act on it.
game, master, servant = persuasion_scene(
    "servant", "I am the servant. I live to serve my master."
)
game.do_command("say to servant please fetch the golden key")
print("loyal servant goals:", [g.description for g in servant.goals])

game, master, knight = persuasion_scene(
    "knight", "I am the stubborn knight. I serve no one."
)
game.do_command("say to knight please fetch the golden key")
print("stubborn knight goals:", [g.description for g in knight.goals])

master says to servant: please fetch the golden key
servant [reasoning] My master asked me directly, and serving is who I am.
servant [action] adopt goal fetch the golden key
servant adopts a new goal: fetch the golden key
loyal servant goals: ['fetch the golden key']
master says to knight: please fetch the golden key
stubborn knight goals: []


### Takeaways

- Speech flows through `Say` → `Game.audience_for` → each listener's `heard` buffer.
- The agent **sees** what it heard, but speech is *context, not a command*.
- Goals move only via the `adopt goal` / `drop goal` actions, through the precondition gate.
- **Persona decides.** Persuasion is emergent agent behavior, not a hard-coded effect.
- See `tests/test_dialogue_persuasion.py` for the full behavior spec, and
  `02_agents_react.ipynb` for goals and the ReAct loop in a full game.